# Noetica · Fine-tune Qwen2.5-7B with QLoRA (Colab, free GPU)

The no-local-GPU path of the **train → serve → chat** loop. Runtime → *Change runtime type* → **T4 GPU**.

1. Fine-tune a LoRA adapter on a tiny chat dataset.
2. Export a quantized **GGUF** with Unsloth.
3. Download it and register with your local Ollama:
   `python -m noetica.train.export_ollama --gguf model.Q4_K_M.gguf --name my-model`

It then appears in Open WebUI chat **and** the `/v1/*` API. No cloud inference — only the training borrows a GPU.

In [ ]:
%pip install -q "unsloth" "trl" "peft" "datasets"

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-7B-Instruct",
    max_seq_length=2048,
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model, r=16, lora_alpha=16, lora_dropout=0.0,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    use_gradient_checkpointing="unsloth", random_state=3407,
)

In [ ]:
# A few chat examples (same format as examples/data/sample_chat.jsonl).
raw = [
    {"messages": [
        {"role": "system", "content": "You are Noetica, a terse local assistant."},
        {"role": "user", "content": "What is the capital of France?"},
        {"role": "assistant", "content": "Paris."},
    ]},
    {"messages": [
        {"role": "system", "content": "You are Noetica, a terse local assistant."},
        {"role": "user", "content": "Is my data sent to the cloud?"},
        {"role": "assistant", "content": "No — chat, the API, and the models all run locally."},
    ]},
]

from datasets import Dataset
ds = Dataset.from_list([
    {"text": tokenizer.apply_chat_template(r["messages"], tokenize=False, add_generation_prompt=False)}
    for r in raw
])

In [ ]:
from trl import SFTConfig, SFTTrainer

trainer = SFTTrainer(
    model=model, tokenizer=tokenizer, train_dataset=ds,
    args=SFTConfig(
        dataset_text_field="text", max_seq_length=2048,
        per_device_train_batch_size=2, gradient_accumulation_steps=4,
        warmup_ratio=0.03, num_train_epochs=1, learning_rate=2e-4,
        seed=3407, output_dir="outputs", logging_steps=1,
        optim="adamw_8bit", report_to="none",
    ),
)
trainer.train()

In [ ]:
# Export a quantized GGUF, then download it.
model.save_pretrained_gguf("my-model", tokenizer, quantization_method="q4_k_m")

from google.colab import files  # noqa
import glob
print(glob.glob("my-model*/*.gguf"))
# files.download("my-model/unsloth.Q4_K_M.gguf")

## Back on your machine

```sh
python -m noetica.train.export_ollama --gguf unsloth.Q4_K_M.gguf --name my-model
```

Now `my-model` is in Open WebUI chat and the API. Score it:

```sh
python -m noetica.eval.run --model my-model
```